# Análise Exploratória de Dados — Crimes e violência sexual

Este notebook realiza uma EDA do conjunto **Crime Trends and Operations of Criminal Justice Systems (UN-CTS)**, com foco nos registros anuais de violência sexual por país/território entre 2003 e 2011.

**Perguntas orientadoras**

1. Qual é a estrutura e a qualidade dos dados?
2. Como se distribuem a contagem de ocorrências (`sexual violence`) e a taxa (`rate`)?
3. Os valores extremos parecem erros ou heterogeneidade real entre países?
4. Como a taxa se relaciona com a contagem, o país e o ano?

> A análise é descritiva e exploratória. Diferenças observadas não demonstram causalidade e comparações internacionais exigem cautela, pois definições legais, registro e capacidade institucional podem variar.

## 0. Bibliotecas, configuração e carregamento

O código procura primeiro o CSV na pasta do notebook ou no diretório atual. Se ele não estiver disponível, tenta baixar a base pública do Kaggle. Assim, o notebook funciona tanto com o arquivo entregue junto quanto em ambientes como o Colab com acesso à internet.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

COR_NEUTRA = "#b8b8b8"
COR_DESTAQUE = "#432c5e"
COR_SECUNDARIA = "#c47f6a"
COR_ALERTA = "#b33a3a"

ARQUIVO = "crime-trends-and-operations-of-criminal-justice-systems-un-cts-csv-1.csv"

candidatos = [
    Path.cwd() / ARQUIVO,
    Path("/content") / ARQUIVO,
]
if "__file__" in globals():
    candidatos.insert(0, Path(__file__).resolve().parent / ARQUIVO)
csv_path = next((p for p in candidatos if p.exists()), None)

if csv_path is None:
    try:
        import kagglehub
    except ImportError as exc:
        raise FileNotFoundError(
            f"Coloque '{ARQUIVO}' na mesma pasta do notebook ou instale kagglehub."
        ) from exc
    pasta = Path(kagglehub.dataset_download("melissamonfared/crime-trends-and-operations"))
    csv_path = pasta / ARQUIVO

df = pd.read_csv(csv_path)
print(f"Arquivo: {csv_path}")
print(f"Dimensão: {df.shape[0]} linhas × {df.shape[1]} colunas")
display(df.head())

## 1. Princípio estrutural

Cada linha representa a observação de um país/território em um ano. As variáveis são:

- `country/territory`: país ou território;
- `date`: ano de referência;
- `sexual violence`: número absoluto de ocorrências registradas;
- `rate`: taxa padronizada registrada na fonte.

### Etapas não aplicáveis

- **Renomear ou criar variável-alvo:** não foi necessário. A base não é de classificação e não possui um desfecho binário equivalente ao exemplo clínico usado em aula.
- **Excluir colunas:** as quatro variáveis têm função clara na EDA; nenhuma foi considerada irrelevante.
- **Converter `date` para data completa:** a variável contém somente anos inteiros de 2003 a 2011. Mantê-la como inteiro facilita agrupamentos e gráficos temporais sem perda de informação.

In [ ]:
resumo_estrutura = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "não_nulos": df.notna().sum(),
    "ausentes": df.isna().sum(),
    "únicos": df.nunique(dropna=True),
})
display(resumo_estrutura)

print(f"Linhas duplicadas completas: {df.duplicated().sum()}")
print(f"Duplicatas de país/ano: {df.duplicated(['country/territory', 'date']).sum()}")
print(f"Período: {df['date'].min()}–{df['date'].max()}")
print(f"Países/territórios: {df['country/territory'].nunique()}")

**Interpretação.** A base tem 918 combinações únicas de país/ano, 102 países/territórios e nove anos. Não há duplicatas completas nem repetição de uma mesma combinação país/ano, portanto não foi necessário excluir linhas duplicadas. Embora `date` seja numérica no arquivo, seu papel analítico é temporal, não o de uma medida contínua comum.

## 2. Dados ausentes

Primeiro são calculadas contagens e porcentagens. Em seguida, verifica-se se as ausências nas duas medidas ocorrem juntas e como a disponibilidade varia por ano.

In [ ]:
ausentes = pd.DataFrame({
    "n": df.isna().sum(),
    "percentual": (df.isna().mean() * 100).round(1),
})
display(ausentes)

padrao_ausencia = pd.crosstab(
    df["sexual violence"].isna(),
    df["rate"].isna(),
    rownames=["contagem ausente"],
    colnames=["taxa ausente"],
)
display(padrao_ausencia)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(padrao_ausencia, annot=True, fmt="d", cmap="magma", ax=axes[0])
axes[0].set_title("Padrão conjunto de ausências")

disponibilidade_ano = df.groupby("date").agg(
    linhas=("country/territory", "size"),
    contagem_disponível=("sexual violence", "count"),
    taxa_disponível=("rate", "count"),
)
disponibilidade_ano[["contagem_disponível", "taxa_disponível"]].plot(
    marker="o", ax=axes[1], color=[COR_SECUNDARIA, COR_DESTAQUE]
)
axes[1].set_title("Observações disponíveis por ano")
axes[1].set_xlabel("Ano")
axes[1].set_ylabel("Número de observações")
axes[1].legend(["Contagem", "Taxa"])
plt.tight_layout()
plt.show()

**Interpretação e decisão.** Há 272 ausências na contagem (29,6%) e 313 na taxa (34,1%). Todas as 272 linhas sem contagem também estão sem taxa; existem ainda 41 linhas com contagem, mas sem taxa. Esse padrão, somado à variação de cobertura ao longo dos anos, é compatível com diferenças de disponibilidade ou de processamento dos registros — portanto, não é prudente assumir MCAR nem inventar valores por média/mediana.

O conjunto mestre é preservado. Cada análise usa apenas as linhas que possuem as variáveis necessárias (`dropna` local). Essa exclusão por análise é mais transparente que uma remoção global de 313 linhas e evita imputar uma taxa que depende de população e metodologia não presentes na base.

In [ ]:
df_original = df.copy()
df_rate = df.dropna(subset=["rate"]).copy()
df_pares = df.dropna(subset=["sexual violence", "rate"]).copy()

print(f"Base original: {len(df_original)} linhas")
print(f"Disponíveis para análises de taxa: {len(df_rate)} linhas")
print(f"Pares completos contagem–taxa: {len(df_pares)} linhas")

## 3. Princípio distribucional

`sexual violence` e `rate` são quantitativas. A primeira é uma contagem absoluta; a segunda permite comparação relativa, embora diferenças de definição e notificação entre países permaneçam relevantes.

In [ ]:
VARS_QUANT = ["sexual violence", "rate"]

desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()
display(
    desc[["count", "mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for j, col in enumerate(VARS_QUANT):
    sns.histplot(df[col].dropna(), kde=True, color=COR_DESTAQUE, ax=axes[0, j])
    axes[0, j].axvline(df[col].median(), color=COR_ALERTA, linestyle="--", label="Mediana")
    axes[0, j].set_title(f"Distribuição de {col}")
    axes[0, j].legend()

    sns.histplot(np.log1p(df[col].dropna()), kde=True, color=COR_SECUNDARIA, ax=axes[1, j])
    axes[1, j].set_title(f"Escala log1p — {col}")
    axes[1, j].set_xlabel(f"log(1 + {col})")

plt.tight_layout()
plt.show()

**Interpretação.** As duas medidas têm cauda longa à direita. Para `rate`, a média (31,17) é quase o dobro da mediana (16,35), a assimetria é 2,90 e a curtose é 13,71. Isso mostra que poucas observações muito altas influenciam fortemente a média. A escala `log1p` é usada apenas para visualização; os valores originais são mantidos para preservar a unidade e a interpretação.

### Verificação da normalidade de `rate`

O teste de Shapiro–Wilk e o QQ-plot são usados apenas na análise univariada da distribuição. Eles não são testes bivariados de comparação entre grupos.

In [ ]:
stat_sw, p_sw = stats.shapiro(df_rate["rate"])
print(f"Shapiro–Wilk: estatística={stat_sw:.4f}; p-valor={p_sw:.3e}")
print("Conclusão descritiva: há evidência contra a normalidade." if p_sw < 0.05 else "Não houve evidência suficiente contra a normalidade.")

fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df_rate["rate"], dist="norm", plot=ax)
ax.set_title("QQ-plot de rate em relação à Normal")
plt.tight_layout()
plt.show()

**Interpretação.** O p-valor é muito menor que 0,05 e o QQ-plot se afasta da reta, sobretudo na cauda superior. As evidências concordam com a forte assimetria observada. Por isso, medianas, intervalos interquartis e correlação de Spearman recebem maior destaque nas seções seguintes.

## 4. Outliers e decisão de tratamento

A regra do IQR identifica observações estatisticamente extremas. Ela é um sinal para investigação, não uma ordem automática de exclusão.

In [ ]:
def resumo_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mascara = (s < inferior) | (s > superior)
    return pd.Series({
        "limite_inferior": inferior,
        "limite_superior": superior,
        "n_outliers": int(mascara.sum()),
        "percentual": 100 * mascara.mean(),
    })

resumo_outliers = pd.DataFrame({col: resumo_iqr(df[col]) for col in VARS_QUANT}).T
display(resumo_outliers.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, col in zip(axes, VARS_QUANT):
    sns.boxplot(
        data=df, y=col, color=COR_NEUTRA,
        flierprops=dict(markerfacecolor=COR_ALERTA, markeredgecolor=COR_ALERTA), ax=ax
    )
    ax.set_title(f"Outliers por IQR — {col}")
plt.tight_layout()
plt.show()

display(
    df.nlargest(10, "rate")[["country/territory", "date", "sexual violence", "rate"]]
    .reset_index(drop=True)
)

**Decisão.** A regra do IQR marca 40 taxas acima de 96,19. As maiores taxas estão associadas a países/territórios e anos identificáveis, e não a códigos impossíveis ou valores negativos. Países pequenos podem combinar contagens moderadas com taxas elevadas; além disso, diferenças reais e diferenças de registro entre países fazem parte do fenômeno estudado. Sem metadados que provem erro, os extremos foram classificados como **estatísticos e plausíveis** e mantidos. Removê-los reduziria justamente a heterogeneidade que a EDA deve revelar.

## 5. Estatística descritiva antes × depois

Não houve imputação nem remoção global de outliers. O “depois” abaixo representa o subconjunto com `rate` observada, usado somente nas análises que dependem dela. A comparação documenta o efeito real da decisão sobre dados ausentes.

In [ ]:
def metricas_rate(dados):
    s = dados["rate"].dropna()
    return pd.Series({
        "linhas_da_base": len(dados),
        "taxas_observadas": len(s),
        "média": s.mean(),
        "mediana": s.median(),
        "desvio_padrão": s.std(),
        "assimetria": s.skew(),
        "curtose": s.kurtosis(),
    })

comparacao = pd.concat(
    {"antes (base completa)": metricas_rate(df_original),
     "depois (taxa disponível)": metricas_rate(df_rate)},
    axis=1,
).T
display(comparacao.round(2))

**Interpretação.** As estatísticas de `rate` são idênticas porque, em ambos os casos, cálculos dessa variável já ignoram seus valores ausentes. O que muda é a unidade de análise: o subconjunto explícito contém 605 linhas válidas. Essa transparência é preferível a afirmar que houve uma “limpeza” capaz de alterar números quando nenhum valor observado foi modificado.

## 6. Frequências e cobertura das variáveis contextuais

`country/territory` é qualitativa nominal e `date` é temporal. Como há 102 países, mostrar todos em barras seria ilegível; são apresentados os 15 com maior cobertura de taxa, com essa seleção explicitamente justificada.

In [ ]:
cobertura_pais = (
    df.groupby("country/territory")
      .agg(linhas=("date", "size"), taxas_disponíveis=("rate", "count"))
      .assign(percentual=lambda x: 100 * x["taxas_disponíveis"] / x["linhas"])
      .sort_values(["taxas_disponíveis", "percentual"], ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["date"].value_counts().sort_index().plot(kind="bar", color=COR_NEUTRA, ax=axes[0])
axes[0].set_title("Linhas da base por ano")
axes[0].set_xlabel("Ano")
axes[0].set_ylabel("Número de linhas")
axes[0].tick_params(axis="x", rotation=0)

cobertura_pais.head(15).sort_values("taxas_disponíveis")["taxas_disponíveis"].plot(
    kind="barh", color=COR_DESTAQUE, ax=axes[1]
)
axes[1].set_title("15 países com maior cobertura de taxa")
axes[1].set_xlabel("Anos com taxa disponível")
plt.tight_layout()
plt.show()

**Interpretação.** A estrutura original contém uma linha por país em cada um dos nove anos, mas a cobertura das medidas é desigual. Isso reforça que tendências agregadas não representam um painel perfeitamente balanceado: a composição de países com dados disponíveis muda ao longo do período.

## 7. Análise bivariada obrigatória

Os testes de hipótese bivariados (ANOVA, Kruskal–Wallis, Mann–Whitney e qui-quadrado) não são executados porque foram definidos como opcionais na atividade. A seção obrigatória é cumprida por tabelas, medidas descritivas e visualizações das relações.

### 7.1 Contagem de ocorrências × taxa

A correlação de Pearson resume associação linear e é sensível aos extremos. Spearman resume associação monotônica por postos e é mais robusta à assimetria. São coeficientes descritivos; não serão calculados p-valores.

In [ ]:
corr_pearson = df_pares[["sexual violence", "rate"]].corr(method="pearson").iloc[0, 1]
corr_spearman = df_pares[["sexual violence", "rate"]].corr(method="spearman").iloc[0, 1]
print(f"Pares completos: {len(df_pares)}")
print(f"Correlação de Pearson:  {corr_pearson:.3f}")
print(f"Correlação de Spearman: {corr_spearman:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df_pares, x="sexual violence", y="rate", alpha=0.55, color=COR_DESTAQUE, ax=axes[0])
axes[0].set_title("Contagem × taxa — escala original")

sns.scatterplot(data=df_pares, x="sexual violence", y="rate", alpha=0.55, color=COR_SECUNDARIA, ax=axes[1])
axes[1].set_xscale("symlog", linthresh=1)
axes[1].set_yscale("symlog", linthresh=1)
axes[1].set_title("Contagem × taxa — escala logarítmica")
plt.tight_layout()
plt.show()

**Interpretação.** Pearson é fraca (0,12), enquanto Spearman é moderada e positiva (0,44). A diferença sugere uma relação monotônica que não é bem descrita por uma linha reta e sofre influência de valores extremos. Isso é coerente com o fato de a taxa ajustar a contagem pela população: países populosos podem ter muitas ocorrências e taxa moderada, enquanto países pequenos podem ter contagem menor e taxa alta. Não se deve interpretar a associação como causal.

### 7.2 País × taxa

Para evitar selecionar países apenas porque têm um ano extremo, entram no ranking aqueles com pelo menos cinco taxas observadas. O gráfico mostra os dez com maior mediana nesse grupo.

In [ ]:
resumo_pais = df_rate.groupby("country/territory")["rate"].agg(["count", "median", "mean", "max"])
top_paises = resumo_pais.query("count >= 5").nlargest(10, "median").index
ordem = resumo_pais.loc[top_paises].sort_values("median", ascending=False).index

display(resumo_pais.loc[ordem].round(2))

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_rate[df_rate["country/territory"].isin(ordem)],
            x="country/territory", y="rate", order=ordem, color=COR_NEUTRA)
sns.stripplot(data=df_rate[df_rate["country/territory"].isin(ordem)],
             x="country/territory", y="rate", order=ordem,
             color=COR_DESTAQUE, alpha=0.75, size=4)
plt.title("Distribuição da taxa — 10 maiores medianas (mínimo de 5 anos)")
plt.xlabel("")
plt.ylabel("Taxa")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Interpretação.** Há grande heterogeneidade entre países. Bahamas, Saint Vincent and the Grenadines, Grenada, Suécia e Costa Rica aparecem com medianas acima de 130 entre os países com pelo menos cinco anos observados. A posição no gráfico pode refletir tanto prevalência quanto diferenças legais, de notificação e de capacidade de registro; portanto, não é um ranking direto de segurança nem uma avaliação causal.

### 7.3 Ano × taxa

A mediana anual é acompanhada pelo intervalo interquartil e pelo número de observações. O IQR reduz a influência das caudas longas, mas a mudança na cobertura de países deve continuar sendo considerada.

In [ ]:
taxa_ano = df_rate.groupby("date")["rate"].agg(
    n="count", q1=lambda s: s.quantile(0.25), mediana="median",
    q3=lambda s: s.quantile(0.75), média="mean"
)
display(taxa_ano.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(taxa_ano.index, taxa_ano["mediana"], marker="o", color=COR_DESTAQUE, label="Mediana")
ax.fill_between(taxa_ano.index, taxa_ano["q1"], taxa_ano["q3"], color=COR_SECUNDARIA, alpha=0.25, label="IQR")
ax.set_title("Taxa por ano — mediana e intervalo interquartil")
ax.set_xlabel("Ano")
ax.set_ylabel("Taxa")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretação.** A mediana passa de 12,69 em 2003 para aproximadamente 20,82 em 2011, com valor próximo em 2010 (21,14). Porém, a cobertura sobe de 26 observações em 2003 para 90 em 2008 e depois cai para 68 em 2011. Assim, o gráfico sugere aumento no fim da série, mas não permite separar mudança real de taxa, mudança de registro e mudança na composição dos países observados.

### 7.4 País × ano: disponibilidade da taxa

Esta visualização cruza o atributo qualitativo país com o tempo. São usados os 15 países com maior quantidade de anos observados para manter legibilidade.

In [ ]:
paises_cobertura = cobertura_pais.head(15).index
matriz_disp = (
    df[df["country/territory"].isin(paises_cobertura)]
      .assign(disponível=df["rate"].notna().astype(int))
      .pivot(index="country/territory", columns="date", values="disponível")
)

plt.figure(figsize=(11, 6))
sns.heatmap(matriz_disp, cmap=["#eeeeee", COR_DESTAQUE], cbar=False,
            linewidths=0.5, linecolor="white", annot=True, fmt=".0f")
plt.title("Disponibilidade da taxa por país e ano (1 = disponível)")
plt.xlabel("Ano")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Interpretação.** Mesmo entre países de melhor cobertura, nem todos possuem os mesmos anos disponíveis. O padrão confirma que análises temporais agregadas precisam informar o número de observações e evitar conclusões que suponham um painel completo.

## 8. Visualização multivariada

O gráfico combina três dimensões: contagem, taxa e ano. Ele substitui o correlograma do exemplo clínico, que seria inadequado aqui porque há somente duas medidas quantitativas de interesse e uma delas já é uma taxa derivada da contagem e da população.

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=df_pares, x="sexual violence", y="rate", hue="date",
    palette="viridis", alpha=0.7, s=45
)
plt.xscale("symlog", linthresh=1)
plt.yscale("symlog", linthresh=1)
plt.title("Contagem × taxa, com o ano representado pela cor")
plt.xlabel("Ocorrências registradas (escala log)")
plt.ylabel("Taxa (escala log)")
plt.legend(title="Ano", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Interpretação.** As cores dos anos se sobrepõem amplamente, sem separação temporal simples na relação entre contagem e taxa. A dispersão vertical para contagens semelhantes reforça o papel do denominador populacional e de diferenças entre países. Seriam necessários dados de população, definições de registro e cobertura institucional para aprofundar essa relação.

## 9. Síntese: principais conclusões

1. **Cobertura incompleta e estruturada:** 29,6% das contagens e 34,1% das taxas estão ausentes; todas as linhas sem contagem também não têm taxa. A cobertura varia por ano, por isso não foi assumido que os dados faltam completamente ao acaso.
2. **Distribuições muito assimétricas:** `rate` apresenta mediana de 16,35, média de 31,17, assimetria de 2,90 e cauda superior longa. Medianas e IQRs são resumos mais representativos que a média isolada.
3. **Extremos foram mantidos:** 40 taxas são outliers pela regra do IQR, mas estão ligadas a países e anos válidos. Sem evidência de erro, sua exclusão apagaria heterogeneidade relevante.
4. **Contagem e taxa não são equivalentes:** a associação é fraca em Pearson (0,12) e moderada em Spearman (0,44). A taxa depende do denominador populacional e permite que contagens semelhantes correspondam a níveis muito diferentes.
5. **Diferenças entre países e anos exigem cautela:** países apresentam medianas bastante distintas e a mediana anual é maior em 2010–2011 que no início da série. Contudo, cobertura, notificação e composição da amostra também mudam, impedindo uma interpretação causal ou um ranking simples.

## 10. Justificativas finais das etapas não executadas

- Não foi criada variável-alvo, pois o objetivo é exploratório, não preditivo.
- Não foram removidas colunas ou duplicatas, porque as quatro colunas são úteis e não há duplicação.
- Não houve imputação: preencher contagens ou taxas sem população e sem informação sobre o mecanismo de ausência produziria precisão artificial.
- Outliers não foram removidos nem substituídos: são plausíveis e pertencem ao fenômeno analisado.
- Não foi criada matriz de correlação extensa ou correlograma: existem apenas duas medidas quantitativas de interesse, e uma grande matriz não acrescentaria informação.
- Não foi realizada análise de *fairness* por sexo/idade: a base é agregada por país/ano e não contém atributos individuais protegidos.
- Os testes estatísticos bivariados foram omitidos por serem opcionais na atividade. A análise bivariada em si foi realizada e interpretada nas quatro subseções anteriores.

## 11. Declaração de uso de IA

**Ferramenta utilizada:** Codex, da OpenAI.

**Aplicação:** apoio na revisão e reorganização do notebook, correção de código, adaptação do roteiro clínico originalmente usado em aula para a base UN-CTS, escolha de visualizações e redação inicial das interpretações.

**Sugestões aceitas:** remoção de referências clínicas incompatíveis; tratamento de `sexual violence` como contagem quantitativa; uso de exclusão local de ausentes; preservação de outliers plausíveis; análises bivariadas por contagem, taxa, país e ano; e inclusão explícita de limitações.

**Sugestões modificadas:** o roteiro da aula foi mantido como estrutura geral, mas as técnicas foram ajustadas às quatro variáveis disponíveis e ao caráter agregado da base. As conclusões foram conferidas após a execução do código.

**Sugestões rejeitadas:** criação de alvo binário, imputação sem base substantiva, remoção automática de extremos, testes bivariados opcionais e análises de atributos individuais inexistentes. Essas opções foram rejeitadas por não serem adequadas ao conjunto de dados ou por não acrescentarem informação confiável.

> A responsabilidade pela revisão, interpretação e entrega final permanece com o estudante.